# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [2]:
%uv pip install -q torch transformers datasets peft accelerate sentencepiece scikit-learn huggingface_hub hf_transfer trl

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

In [6]:
import os, json
from datasets import Dataset

data_dir = 'splits'
formatted_data = []
skipped = 0

prompt_template = """උපදෙස්: ඔබ සිංහල ප්‍රශ්න-පිළිතුරු සහායකයෙකි. පහත සපයා ඇති තොරතුරු (Context) පමණක් භාවිතා කරමින් ප්‍රශ්නයට නිවැරදි, සෘජු, කෙටි පිළිතුරක් දෙන්න. Context තුළ පිළිතුර නොමැති නම් හෝ ප්‍රමාණවත් තොරතුරු නොමැති නම්, “මෙම ප්‍රශ්නයට පිළිතුරු දීමට ප්‍රමාණවත් තොරතුරු නොමැත.” යනුවෙන් පමණක් පිළිතුරු දෙන්න. Context වලින් පිටත දැනුම, අනුමාන, හෝ අමතර විස්තර භාවිතා නොකරන්න.\n\nතොරතුරු (Context): {context}\nප්‍රශ්නය: {question}\nපිළිතුර: [{answer}]"""

for filename in os.listdir(data_dir):
    if filename == "train.jsonl":
        filepath = os.path.join(data_dir, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if line.strip():
                    try:
                        item = json.loads(line)
                        text = prompt_template.format(
                            context=item.get("context", ""),
                            question=item.get("question", ""),
                            answer=item.get("answer", "")
                        )
                        formatted_data.append({"text": text})
                    except json.JSONDecodeError as e:
                        skipped += 1
                        print(f"Skipping {filename} line {i+1}: {e}")

print(f"\nTotal loaded: {len(formatted_data)} | Skipped: {skipped}")
dataset = Dataset.from_list(formatted_data)
print("\n--- Example Format ---\n")
print(dataset[0]["text"])

Skipping 6.jsonl line 69: Expecting ',' delimiter: line 1 column 73 (char 72)
Skipping 6.jsonl line 195: Expecting ',' delimiter: line 2 column 1 (char 281)
Skipping 11.jsonl line 35: Expecting ',' delimiter: line 1 column 122 (char 121)
Skipping 11.jsonl line 70: Invalid \escape: line 1 column 263 (char 262)
Skipping 11.jsonl line 101: Expecting ',' delimiter: line 1 column 285 (char 284)
Skipping 11.jsonl line 102: Expecting ',' delimiter: line 1 column 38 (char 37)
Skipping 11.jsonl line 336: Expecting ',' delimiter: line 1 column 16 (char 15)
Skipping 11.jsonl line 340: Expecting ',' delimiter: line 1 column 188 (char 187)
Skipping 11.jsonl line 345: Expecting ',' delimiter: line 1 column 16 (char 15)
Skipping 11.jsonl line 346: Expecting ',' delimiter: line 1 column 16 (char 15)
Skipping 11.jsonl line 360: Expecting ',' delimiter: line 1 column 16 (char 15)
Skipping 11.jsonl line 408: Expecting ',' delimiter: line 1 column 351 (char 350)
Skipping 8.jsonl line 13: Expecting ',' del

In [7]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model_id = "meta-llama/Llama-3.2-1B"
adapter_id = "isji/sinllama-1b-cpt"
tokenizer_id = "polyglots/Extended-Sinhala-LLaMA"

print("Loading Sinhala tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(tokenizer_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base Llama model...")
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
model.resize_token_embeddings(len(tokenizer))

print("Loading and merging CPT adapter...")
model = PeftModel.from_pretrained(model, adapter_id)
model = model.merge_and_unload()
print("Model merged and ready!")

Loading Sinhala tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Loading base Llama model...


...

adapter_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


adapter_model.safetensors:   0%|          | 0.00/3.45G [00:00<?, ?B/s]

Model merged and ready!


/usr/local/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:683: UserWarning: Input and output embeddings are no longer tied after merging. Setting `tie_word_embeddings=False` in the model config.
  warnings.warn(


In [8]:
%uv pip install -U bitsandbytes

Using Python 3.12.6 environment at: /usr/local
Resolved 32 packages in 209ms
⠙ Preparing packages... (0/31)
⠙ Preparing packages... (0/31)
⠙ Preparing packages... (0/31)
typing-extensions ------------------------------     0 B/43.57 KiB
⠙ Preparing packages... (0/31)
typing-extensions ------------------------------ 14.80 KiB/43.57 KiB
⠙ Preparing packages... (0/31)
filelock   ------------------------------     0 B/38.88 KiB
typing-extensions ------------------------------ 14.80 KiB/43.57 KiB
⠙ Preparing packages... (0/31)
filelock   ------------------------------ 14.84 KiB/38.88 KiB
typing-extensions ------------------------------ 14.80 KiB/43.57 KiB
⠙ Preparing packages... (0/31)
filelock   ------------------------------ 14.84 KiB/38.88 KiB
typing-extensions ------------------------------ 14.80 KiB/43.57 KiB
packaging  ------------------------------ 14.81 KiB/97.85 KiB
⠙ Preparing packages... (0/31)
filelock   ------------------------------ 14.84 KiB/38.88 KiB
typing-extensions ------

In [14]:
from peft import LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig

output_dir = "./output_sinllama_1b_qa_suggested"

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.10,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "down_proj", "up_proj"],
    # modules_to_save=["lm_head"] is intentionally disabled for this small-data QA run.
)

training_args = SFTConfig(
    output_dir=output_dir,
    dataset_text_field="text",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    optim="adamw_torch",
    weight_decay=0.01,
    max_grad_norm=1.0,
    report_to="none",
)

tokenizer.model_max_length = 1024
tokenizer.padding_side = "right"

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args,
    peft_config=peft_config,
)

print("Starting QA Fine-Tuning v2...")
trainer.train()

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"\nModel saved to {output_dir}")

Adding EOS to train dataset:   0%|          | 0/1919 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1919 [00:00<?, ? examples/s]

Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Starting QA Fine-Tuning v2...


Step,Training Loss
10,1.765350
20,1.326125
30,0.806159
40,0.665750
50,0.648465
60,0.632060
70,0.602696
80,0.544361
90,0.528011
100,0.521984



Model saved to ./output_sinllama_1b_qa_v2


In [15]:
import warnings
warnings.filterwarnings('ignore')

# Save the final adapter locally
final_adapter_path = os.path.join(output_dir, "qa_lora_model")
trainer.model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"\nQA Adapter saved to {final_adapter_path}")

# Optional: Push to HuggingFace Hub
# repo_id = "isji/sinllama-1b-qa"
# trainer.model.push_to_hub(repo_id)
# tokenizer.push_to_hub(repo_id)
# print(f"Pushed to HuggingFace Hub: {repo_id}")

# --- Quick Inference Test ---
test_context = "ශ්‍රී ලංකාව දකුණු ආසියාවේ පිහිටි දූපත් රාජ්‍යකි. කොළඹ ශ්‍රී ලංකාවේ වාණිජ අගනුවරයි."
test_context2 = "හීලියම් වායුව යනු කුමක්ද?"
test_question = "ශ්‍රී ලංකාවේ වාණිජ අගනුවර කුමක්ද?"

test_prompt = f"""උපදෙස්: පහත සපයා ඇති තොරතුරු (Context) පමණක් භාවිතා කරමින් ප්‍රශ්නයට නිවැරදි, කෙටි පිළිතුරක් දෙන්න.\n\nතොරතුරු (Context): {test_context}\nප්‍රශ්නය: {test_question}\nපිළිතුර: ["""

inputs = tokenizer(test_prompt, return_tensors="pt").to(trainer.model.device)
trainer.model.eval()

with torch.no_grad():
    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        repetition_penalty=1.15,
    )

full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n--- Inference Output ---")
print(full_text)


QA Adapter saved to ./output_sinllama_1b_qa_v2/qa_lora_model

--- Inference Output ---
උපදෙස්: පහත සපයා ඇති තොරතුරු (Context) පමණක් භාවිතා කරමින් ප්‍රශ්නයට නිවැරදි, කෙටි පිළිතුරක් දෙන්න.

තොරතුරු (Context): ශ්‍රී ලංකාව දකුණු ආසියාවේ පිහිටි දූපත් රාජ්‍යකි. කොළඹ ශ්‍රී ලංකාවේ වාණිජ අගනුවරයි.
ප්‍රශ්නය: ශ්‍රී ලංකාවේ වාණිජ අගනුවර කුමක්ද?
පිළිතුර: [කොළඹ ය.]


In [16]:
import warnings
warnings.filterwarnings('ignore')

# Save the final adapter locally
final_adapter_path = os.path.join(output_dir, "qa_lora_model")
trainer.model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"\nQA Adapter saved to {final_adapter_path}")

# --- Inference Helper ---
def run_qa(context, question, max_new_tokens=80):
    prompt = (
        f"උපදෙස්: පහත සපයා ඇති තොරතුරු (Context) පමණක් භාවිතා කරමින් "
        f"ප්‍රශ්නයට නිවැරදි, කෙටි පිළිතුරක් දෙන්න.\n\n"
        f"තොරතුරු (Context): {context}\n"
        f"ප්‍රශ්නය: {question}\n"
        f"පිළිතුර: ["
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)
    trainer.model.eval()
    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.15,
        )
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the answer part
    answer = full_text.split("පිළිතුර: [")[-1].strip()
    return answer

# --- Test Cases ---
tests = [
    # 1. Short context (original)
    {
        "name": "Short – Sri Lanka capital",
        "context": "ශ්‍රී ලංකාව දකුණු ආසියාවේ පිහිටි දූපත් රාජ්‍යකි. කොළඹ ශ්‍රී ලංකාවේ වාණිජ අගනුවරයි.",
        "question": "ශ්‍රී ලංකාවේ වාණිජ අගනුවර කුමක්ද?"
    },
    # 2. Medium context – geography paragraph
    {
        "name": "Medium – Mahaweli River",
        "context": (
            "මහවැලි ගඟ ශ්‍රී ලංකාවේ දිගම ගඟයි. එහි දිග කිලෝමීටර් 335කි. "
            "මෙම ගඟ මධ්‍යම කඳුකරයේ උල්පත් ගෙන නැගෙනහිර වෙරළ දෙසට ගලා යයි. "
            "මහවැලි සංවර්ධන ව්‍යාපෘතිය ශ්‍රී ලංකාවේ විශාලතම ජලාශ ව්‍යාපෘතියයි. "
            "එය 1970 දශකයේදී ආරම්භ කරන ලදී."
        ),
        "question": "මහවැලි ගඟේ දිග කීයද?"
    },
    # 3. Medium context – science
    {
        "name": "Medium – Photosynthesis",
        "context": (
            "ප්‍රකාශ සංශ්ලේෂණය යනු ශාක, ඇල්ගී සහ සමහර බැක්ටීරියා විසින් සූර්යාලෝකය, "
            "ජලය සහ කාබන් ඩයොක්සයිඩ් භාවිතා කරමින් ග්ලූකෝස් නිපදවන ක්‍රියාවලියයි. "
            "මෙම ක්‍රියාවලියේදී ඔක්සිජන් අතුරු ඵලයක් ලෙස නිකුත් වේ. "
            "ප්‍රකාශ සංශ්ලේෂණය සිදු වන්නේ ශාකයේ හරිත පැහැ ද්‍රව්‍යයක් වන ක්ලෝරොෆිල් හරහාය."
        ),
        "question": "ප්‍රකාශ සංශ්ලේෂණයේදී නිකුත් වන්නේ කුමක්ද?"
    },
    # 4. Large context – history passage
    {
        "name": "Large – Kandyan Kingdom",
        "context": (
            "මහනුවර රාජධානිය ශ්‍රී ලංකාවේ අවසාන ස්වාධීන රාජධානියයි. "
            "එය 1469 සිට 1815 දක්වා පැවතිණි. රාජධානියේ අවසාන රජු වූ ශ්‍රී වික්‍රම රාජසිංහ "
            "1815 දී බ්‍රිතාන්‍යයන් සමඟ ඇති කරගත් මහනුවර සම්මුතිය මඟින් බලය භාර දුන්නේය. "
            "මහනුවර රාජධානිය දළදා මාලිගාව සඳහා ප්‍රසිද්ධය. දළදා වහන්සේ "
            "ශ්‍රී ලංකාවේ රාජකීය බලයේ සංකේතය ලෙස සැලකේ. "
            "රාජධානියේ රාජධානිය ලෙස සේවය කළ නගරය වර්තමාන මහනුවර නගරයයි. "
            "ශ්‍රී ලංකාවේ මධ්‍යම පළාතේ පිහිටි මෙම නගරය සමුද්‍ර මට්ටමින් මීටර් 500ක් පමණ උසින් පිහිටා ඇත."
        ),
        "question": "මහනුවර රාජධානියේ අවසාන රජු කව්ද?"
    },
    # 5. Unanswerable – context doesn't contain the answer
    {
        "name": "Unanswerable – out of context",
        "context": (
            "ඇල්බට් අයින්ස්ටයින් 1879 මාර්තු 14 වැනිදා ජර්මනියේ උල්ම් නගරයේ උපන්නේය. "
            "ඔහු සාපේක්ෂතා විශේෂ සහ සාමාන්‍ය න්‍යාය සකස් කිරීම සඳහා ප්‍රසිද්ධය. "
            "1921 දී ඔහුට භෞතික විද්‍යාව සඳහා නොබෙල් ත්‍යාගය ලබා දෙන ලදී."
        ),
        "question": "අයින්ස්ටයින්ගේ දියණියගේ නම කුමක්ද?"  # Not in context
    },
]

# --- Run All Tests ---
trainer.model.eval()
print("\n" + "="*60)
print("QA INFERENCE TESTS")
print("="*60)

for test in tests:
    print(f"\n📌 [{test['name']}]")
    print(f"   Context : {test['context'][:80]}{'...' if len(test['context']) > 80 else ''}")
    print(f"   Question: {test['question']}")
    answer = run_qa(test["context"], test["question"])
    print(f"   Answer  : {answer}")
    print("-"*60)


QA Adapter saved to ./output_sinllama_1b_qa_v2/qa_lora_model

QA INFERENCE TESTS

📌 [Short – Sri Lanka capital]
   Context : ශ්‍රී ලංකාව දකුණු ආසියාවේ පිහිටි දූපත් රාජ්‍යකි. කොළඹ ශ්‍රී ලංකාවේ වාණිජ අගනුවරය...
   Question: ශ්‍රී ලංකාවේ වාණිජ අගනුවර කුමක්ද?
   Answer  : කොළඹ ය.]
------------------------------------------------------------

📌 [Medium – Mahaweli River]
   Context : මහවැලි ගඟ ශ්‍රී ලංකාවේ දිගම ගඟයි. එහි දිග කිලෝමීටර් 335කි. මෙම ගඟ මධ්‍යම කඳුකරයේ...
   Question: මහවැලි ගඟේ දිග කීයද?
   Answer  : අක්කර 335 කි.]
------------------------------------------------------------

📌 [Medium – Photosynthesis]
   Context : ප්‍රකාශ සංශ්ලේෂණය යනු ශාක, ඇල්ගී සහ සමහර බැක්ටීරියා විසින් සූර්යාලෝකය, ජලය සහ කා...
   Question: ප්‍රකාශ සංශ්ලේෂණයේදී නිකුත් වන්නේ කුමක්ද?
   Answer  : සපයා ඇති සන්දර්භය මත පදනම්ව මෙය පිළිතුරු දිය නොහැක.]
------------------------------------------------------------

📌 [Large – Kandyan Kingdom]
   Context : මහනුවර රාජධානිය ශ්‍රී ලංකාවේ අවසාන ස්වාධීන රාජධානියයි. එය 146

In [18]:
history_tests = [
    # Ancient history
    {
        "name": "Ancient – Anuradhapura Kingdom",
        "context": (
            "අනුරාධපුර රාජධානිය ශ්‍රී ලංකාවේ පළමු ප්‍රධාන රාජධානියයි. "
            "එය ක්‍රිස්තු පූර්ව 377 සිට ක්‍රිස්තු වර්ෂ 1017 දක්වා පැවතිණි. "
            "අනුරාධපුරය ශ්‍රී ලංකාවේ උතුරු මධ්‍යම පළාතේ පිහිටා ඇත. "
            "මෙම රාජධානිය පාලනය කළ පළමු රජු පණ්ඩුකාභය රජු බව සැලකේ. "
            "අනුරාධපුර රාජධානිය බෞද්ධ ශිෂ්ටාචාරයේ කේන්ද්‍රස්ථානයක් විය."
        ),
        "question": "අනුරාධපුර රාජධානිය පාලනය කළ පළමු රජු කව්ද?"
    },
    # Colonial history
    {
        "name": "Colonial – Portuguese arrival",
        "context": (
            "පෘතුගීසීන් 1505 දී ශ්‍රී ලංකාවට පැමිණියේ ලොරෙන්සෝ ද අල්මේදා නායකත්වයෙනි. "
            "ඔවුන් මුලින්ම ගාල්ල වරායට පැමිණ පසුව කොළඹ බලකොටුව ඉදිකළහ. "
            "පෘතුගීසීන් ශ්‍රී ලංකාව යටත් කර ගත්තේ වෙළඳ අරමුණු ඇතිවය. "
            "ඔවුන් ශ්‍රී ලංකාවේ පැවැතියේ 1505 සිට 1658 දක්වාය. "
            "දෙමළ සහ සිංහල ජනතාව මත කතෝලික ආගම පතුරවා හැරීම ද ඔවුන්ගේ අරමුණක් විය."
        ),
        "question": "පෘතුගීසීන් ශ්‍රී ලංකාවට පැමිණියේ කවදාද?"
    },
    # Colonial – Dutch
    {
        "name": "Colonial – Dutch period",
        "context": (
            "ලන්දේසීන් 1658 දී පෘතුගීසීන්ව පරාජය කොට ශ්‍රී ලංකාවේ බලය අත්පත් කරගත්හ. "
            "ලන්දේසීන් ශ්‍රී ලංකාවේ ඉන්දියානු සාගර වෙළඳාම පාලනය කිරීමට උනන්දු වූහ. "
            "ඔවුන් ශ්‍රී ලංකාවේ නීති ක්‍රමය, ඇල් පද්ධතිය සහ වාස්තු විද්‍යාව කෙරෙහි දායකත්වය සැලසූහ. "
            "ලන්දේසීන් 1658 සිට 1796 දක්වා ශ්‍රී ලංකාව පාලනය කළහ."
        ),
        "question": "ලන්දේසීන් ශ්‍රී ලංකාවේ බලය අත්පත් කරගත්තේ කවදාද?"
    },
    # Independence
    {
        "name": "Modern – Independence",
        "context": (
            "ශ්‍රී ලංකාව 1948 පෙබරවාරි 4 වැනිදා බ්‍රිතාන්‍ය යටත් විජිත පාලනයෙන් නිදහස ලැබීය. "
            "නිදහස ලැබූ පළමු අග්‍රාමාත්‍යවරයා වූයේ ඩී.එස්. සේනානායක මහතාය. "
            "එවකට රටේ නම සිලෝන් ලෙස හැඳින්වූ අතර 1972 දී ශ්‍රී ලංකා ජනරජය ලෙස නම් කෙරිණි. "
            "නිදහස් දිනය සමරනු ලබන්නේ පෙබරවාරි 4 වැනිදාය."
        ),
        "question": "ශ්‍රී ලංකාවේ පළමු අග්‍රාමාත්‍යවරයා කව්ද?"
    },
    # Landmark – Sigiriya
    {
        "name": "Landmark – Sigiriya",
        "context": (
            "සීගිරිය ශ්‍රී ලංකාවේ මධ්‍යම පළාතේ පිහිටි පුරාණ ගල් බලකොටුවකි. "
            "එය ඉදිකළේ රජ කාශ්‍යප විසිනි. ඔහු ක්‍රිස්තු වර්ෂ 477 සිට 495 දක්වා රජකම් කළේය. "
            "සීගිරිය යූනෙස්කෝ ලෝක උරුම අඩවියක් ලෙස 1982 දී නම් කෙරිණි. "
            "ගල්කුළේ උස මීටර් 180කි. සීගිරිය රූකථා ද ලොව ප්‍රසිද්ධය."
        ),
        "question": "සීගිරිය ඉදිකළේ කවුද?"
    },
    # Unanswerable – history domain
    {
        "name": "Unanswerable – history domain",
        "context": (
            "පොළොන්නරු රාජධානිය ශ්‍රී ලංකාවේ දෙවන විශාලතම පුරාණ රාජධානියයි. "
            "එය ක්‍රිස්තු වර්ෂ 1070 සිට 1293 දක්වා පැවතිණි. "
            "පරාක්‍රමබාහු පළමුවැනි රජු මෙම රාජධානියේ බලවත්ම රජු ලෙස සැලකේ."
        ),
        "question": "පොළොන්නරු රාජධානියේ රාජකීය මාලිගාවේ උස කීයද?"  # Not in context
    },
]

# --- Run History Tests ---
trainer.model.eval()
print("\n" + "="*60)
print("HISTORY QA INFERENCE TESTS")
print("="*60)

for test in history_tests:
    print(f"\n📌 [{test['name']}]")
    print(f"   Context : {test['context'][:80]}{'...' if len(test['context']) > 80 else ''}")
    print(f"   Question: {test['question']}")
    answer = run_qa(test["context"], test["question"])
    print(f"   Answer  : {answer}")
    print("-"*60)


HISTORY QA INFERENCE TESTS

📌 [Ancient – Anuradhapura Kingdom]
   Context : අනුරාධපුර රාජධානිය ශ්‍රී ලංකාවේ පළමු ප්‍රධාන රාජධානියයි. එය ක්‍රිස්තු පූර්ව 377 ...
   Question: අනුරාධපුර රාජධානිය පාලනය කළ පළමු රජු කව්ද?
   Answer  : පණ්ඩුකාභය රජතුමා ය.]
------------------------------------------------------------

📌 [Colonial – Portuguese arrival]
   Context : පෘතුගීසීන් 1505 දී ශ්‍රී ලංකාවට පැමිණියේ ලොරෙන්සෝ ද අල්මේදා නායකත්වයෙනි. ඔවුන් ම...
   Question: පෘතුගීසීන් ශ්‍රී ලංකාවට පැමිණියේ කවදාද?
   Answer  : 1505 වර්ෂයේදී ය.]
------------------------------------------------------------

📌 [Colonial – Dutch period]
   Context : ලන්දේසීන් 1658 දී පෘතුගීසීන්ව පරාජය කොට ශ්‍රී ලංකාවේ බලය අත්පත් කරගත්හ. ලන්දේසීන...
   Question: ලන්දේසීන් ශ්‍රී ලංකාවේ බලය අත්පත් කරගත්තේ කවදාද?
   Answer  : 1658 දී ය.]
------------------------------------------------------------

📌 [Modern – Independence]
   Context : ශ්‍රී ලංකාව 1948 පෙබරවාරි 4 වැනිදා බ්‍රිතාන්‍ය යටත් විජිත පාලනයෙන් නිදහස ලැබීය. ...
   Question: 

In [19]:
quick_tests = [
    {
        "name": "Q1 – Parakramabahu",
        "context": (
            "පරාක්‍රමබාහු පළමුවැනි රජු 1153 සිට 1186 දක්වා පොළොන්නරු රාජධානිය පාලනය කළේය. "
            "ඔහු ශ්‍රී ලංකාවේ ජල කළමනාකරණ ක්‍රමය නවීකරණය කළ රජකු ලෙස ප්‍රසිද්ධය. "
            "පරාක්‍රම සමුද්‍රය ඉදිකළේ ද ඔහු විසිනි. එය ශ්‍රී ලංකාවේ විශාලතම පුරාණ ජලාශයයි."
        ),
        "question": "පරාක්‍රම සමුද්‍රය ඉදිකළේ කවුද?"
    },
    {
        "name": "Q2 – Dutugemunu",
        "context": (
            "දුටුගැමුණු රජු ක්‍රිස්තු පූර්ව 161 සිට 137 දක්වා රාජ්‍යය පාලනය කළේය. "
            "ඔහු එළාර රජු පරාජය කොට ශ්‍රී ලංකාව එකමුතු කළේය. "
            "රුවන්වැලිසාය ස්තූපය ඉදිකිරීම ආරම්භ කළේ දුටුගැමුණු රජු විසිනි. "
            "ඔහු අනුරාධපුර රාජධානියේ වීරත්වය ගත් රජකු ලෙස ඉතිහාසයේ සටහන් වෙයි."
        ),
        "question": "රුවන්වැලිසාය ස්තූපය ඉදිකිරීම ආරම්භ කළේ කවුද?"
    },
    {
        "name": "Q3 – Buddhism arrival",
        "context": (
            "බුදු දහම ශ්‍රී ලංකාවට ගෙන ආවේ මහින්ද තෙරුන් විසිනි. "
            "එය සිදු වූයේ ක්‍රිස්තු පූර්ව 247 දී දේවානම්පියතිස්ස රජුගේ පාලන සමයේදීය. "
            "මහින්ද තෙරුන් අශෝක අධිරාජයාගේ පුත්‍රයා බව සැලකේ. "
            "බුදු දහම ශ්‍රී ලංකාවේ රාජකීය ආගම බවට පත් වූයේ එතැන් සිටය."
        ),
        "question": "බුදු දහම ශ්‍රී ලංකාවට ගෙන ආවේ කවුද?"
    },
    {
        "name": "Q4 – Vijaya",
        "context": (
            "විජය කුමාරයා ශ්‍රී ලංකාවේ ඉතිහාසයේ සඳහන් පළමු රජු ලෙස සැලකේ. "
            "ඔහු ඉන්දියාවේ සිංහ රාජකුලයෙන් පැවත ආ බව මහාවංශයේ සඳහන් වේ. "
            "විජය ශ්‍රී ලංකාවට පැමිණියේ ක්‍රිස්තු පූර්ව 543 දී බව විශ්වාස කෙරේ. "
            "ඔහු සමඟ 700 දෙනෙකු ශ්‍රී ලංකාවට පැමිණි බව සඳහන් වේ."
        ),
        "question": "විජය ශ්‍රී ලංකාවට පැමිණියේ කවදාද?"
    },
    {
        "name": "Q5 – British takeover",
        "context": (
            "බ්‍රිතාන්‍යයන් 1796 දී ලන්දේසීන්ගෙන් ශ්‍රී ලංකාවේ බලය අත්පත් කරගත්හ. "
            "1815 දී මහනුවර සම්මුතිය අත්සන් කිරීමෙන් මහනුවර රාජධානිය ද ඔවුන් යටතට පත් විය. "
            "බ්‍රිතාන්‍ය පාලනය යටතේ කෝපි, තේ සහ පොල් වගාව ප්‍රවර්ධනය විය. "
            "ශ්‍රී ලංකාව සිලෝන් නමින් බ්‍රිතාන්‍ය යටත් විජිතයක් ලෙස 1948 දක්වා පැවතිණි."
        ),
        "question": "මහනුවර සම්මුතිය අත්සන් කෙරුණේ කවදාද?"
    },
    {
        "name": "Q6 – Mahaweli irrigation",
        "context": (
            "මහවැලි සංවර්ධන ව්‍යාපෘතිය ශ්‍රී ලංකාවේ විශාලතම සංවර්ධන ව්‍යාපෘතියයි. "
            "එය 1970 දශකයේදී ජේ.ආර්. ජයවර්ධන ජනාධිපතිවරයාගේ නායකත්වය යටතේ ආරම්භ කෙරිණි. "
            "මෙම ව්‍යාපෘතිය හරහා විශාල ජල විදුලි බලශක්තියක් ජනනය වේ. "
            "වැව් 30කට අධික ප්‍රමාණයක් සහ ඇල මාර්ග ජාලයක් ගොඩනගන ලදී."
        ),
        "question": "මහවැලි සංවර්ධන ව්‍යාපෘතිය ආරම්භ කෙරුණේ කාගේ නායකත්වය යටතේද?"
    },
    {
        "name": "Q7 – Tooth Relic",
        "context": (
            "දළදා වහන්සේ ශ්‍රී ලංකාවට ගෙන ආවේ හේමමාලා කුමරිය සහ දන්ත කුමාරයා විසිනි. "
            "ඔවුන් එය ඔවුන්ගේ කෙස් රැළිය තුළ සඟවාගෙන ලංකාවට ගෙන ආහ. "
            "එය සිදු වූයේ ක්‍රිස්තු වර්ෂ 4 වැනි සියවසේදීය. "
            "දළදා මාලිගාව දැනට පිහිටා ඇත්තේ මහනුවර නගරයේය."
        ),
        "question": "දළදා වහන්සේ ශ්‍රී ලංකාවට ගෙන ආවේ කවුද?"
    },
    {
        "name": "Q8 – First female PM",
        "context": (
            "සිරිමාවෝ බණ්ඩාරනායක මහත්මිය ලොව පළමු කාන්තා අගමැතිවරිය වූවාය. "
            "ඇය 1960 දී ශ්‍රී ලංකාවේ අගමැතිනිය ලෙස පත් වූවාය. "
            "ඇය ශ්‍රී ලංකා නිදහස් පක්ෂයේ නායිකාවක් ලෙස කටයුතු කළාය. "
            "ඇගේ ස්වාමිපුරුෂයා වූ එස්.ඩබ්.ආර්.ඩී. බණ්ඩාරනායක 1959 දී ඝාතනය කරන ලදී."
        ),
        "question": "සිරිමාවෝ බණ්ඩාරනායක මහත්මිය අගමැතිනිය ලෙස පත් වූයේ කවදාද?"
    },
    {
        "name": "Q9 – Unanswerable",
        "context": (
            "ගාල්ල කොටුව 1588 දී පෘතුගීසීන් විසින් ඉදිකරන ලදී. "
            "පසුව 1663 දී ලන්දේසීන් එය නවීකරණය කළහ. "
            "ගාල්ල කොටුව යූනෙස්කෝ ලෝක උරුම අඩවියක් ලෙස 1988 දී නම් කෙරිණි."
        ),
        "question": "ගාල්ල කොටුවේ ප්‍රධාන දොරටුවේ පළල කීයද?"  # Not in context
    },
    {
        "name": "Q10 – Mahinda Rajapaksa",
        "context": (
            "මහින්ද රාජපක්ෂ මහතා 2005 සිට 2015 දක්වා ශ්‍රී ලංකාවේ ජනාධිපතිවරයා ලෙස සේවය කළේය. "
            "ඔහුගේ පාලන සමයේදී 2009 මැයි මාසයේදී දශක තුනකට අධික ගැටුම අවසන් විය. "
            "ඔහු හම්බන්තොට දිස්ත්‍රික්කයෙන් පාර්ලිමේන්තු මන්ත්‍රීවරයෙකු ලෙස ද කටයුතු කළේය."
        ),
        "question": "ශ්‍රී ලංකාවේ සන්නද්ධ ගැටුම අවසන් වූයේ කවදාද?"
    },
]

# --- Run ---
trainer.model.eval()
print("\n" + "="*60)
print("10 HISTORY QA TESTS")
print("="*60)

for test in quick_tests:
    print(f"\n📌 [{test['name']}]")
    print(f"   Q: {test['question']}")
    answer = run_qa(test["context"], test["question"])
    print(f"   A: {answer}")
    print("-"*60)


10 HISTORY QA TESTS

📌 [Q1 – Parakramabahu]
   Q: පරාක්‍රම සමුද්‍රය ඉදිකළේ කවුද?
   A: ඔහු ය.]
------------------------------------------------------------

📌 [Q2 – Dutugemunu]
   Q: රුවන්වැලිසාය ස්තූපය ඉදිකිරීම ආරම්භ කළේ කවුද?
   A: දුටුගැමුණු රජු විසිනි.]
------------------------------------------------------------

📌 [Q3 – Buddhism arrival]
   Q: බුදු දහම ශ්‍රී ලංකාවට ගෙන ආවේ කවුද?
   A: මහින්ද හිමියන් ය.]
------------------------------------------------------------

📌 [Q4 – Vijaya]
   Q: විජය ශ්‍රී ලංකාවට පැමිණියේ කවදාද?
   A: ක්‍රිස්තු පූර්ව 543 දී ය.]
------------------------------------------------------------

📌 [Q5 – British takeover]
   Q: මහනුවර සම්මුතිය අත්සන් කෙරුණේ කවදාද?
   A: 1796 වර්ෂයේදී ය.]
------------------------------------------------------------

📌 [Q6 – Mahaweli irrigation]
   Q: මහවැලි සංවර්ධන ව්‍යාපෘතිය ආරම්භ කෙරුණේ කාගේ නායකත්වය යටතේද?
   A: ජේ.පී. ජයවර්ධන මහතාගේ නායකත්වයෙන් ය.]
------------------------------------------------------------

📌 [Q7 – Tooth Rel

In [21]:
import re
import unicodedata

def normalize_answer(text: str) -> str:
    if text is None:
        return ""

    text = unicodedata.normalize("NFC", text).strip().lower()

    # Remove trailing punctuation/noise that the model often adds
    text = re.sub(r"[\]\)\}\.,!?;៖:]+$", "", text)
    text = re.sub(r"\s+", " ", text)

    # Remove common copular endings from generated answers
    text = re.sub(r"\s+(ය|යි|යින්|දී|විය|වුනා|වේ)$", "", text)

    return text.strip()


def is_unanswerable(pred: str) -> bool:
    pred_n = normalize_answer(pred)
    keywords = [
        "ප්‍රමාණවත් තොරතුරු නොමැත",
        "තොරතුරු නොමැත",
        "නොදනී",
        "සඳහන් නොවේ",
        "answer cannot be determined",
        "cannot be determined",
        "unknown",
    ]
    return any(k in pred_n for k in keywords)


def tokenize(text: str):
    text = normalize_answer(text)
    return set(text.split())


def keyword_match(pred: str, gold: str, threshold: float = 0.5) -> bool:
    pred_tokens = tokenize(pred)
    gold_tokens = tokenize(gold)

    if not gold_tokens:
        return False

    overlap = pred_tokens.intersection(gold_tokens)
    score = len(overlap) / len(gold_tokens)

    return score >= threshold


def validate_answer(pred: str, gold: str) -> bool:
    pred_n = normalize_answer(pred)
    gold_n = normalize_answer(gold)

    # Handle unanswerable
    if gold_n in {"නොදනී", "තොරතුරු ප්‍රමාණවත් නොමැත", "answer_unavailable"}:
        return is_unanswerable(pred)

    # First try exact match
    if pred_n == gold_n:
        return True

    # Then fallback to keyword match
    return keyword_match(pred, gold, threshold=0.5)


quick_tests = [
    {
        "name": "Q1 – Parakramabahu",
        "context": (
            "පරාක්‍රමබාහු පළමුවැනි රජු 1153 සිට 1186 දක්වා පොළොන්නරු රාජධානිය පාලනය කළේය. "
            "ඔහු ශ්‍රී ලංකාවේ ජල කළමනාකරණ ක්‍රමය නවීකරණය කළ රජකු ලෙස ප්‍රසිද්ධය. "
            "පරාක්‍රම සමුද්‍රය ඉදිකළේ ද ඔහු විසිනි. එය ශ්‍රී ලංකාවේ විශාලතම පුරාණ ජලාශයයි."
        ),
        "question": "පරාක්‍රම සමුද්‍රය ඉදිකළේ කවුද?",
        "expected_answer": "පරාක්‍රමබාහු පළමුවැනි රජු",
    },
    {
        "name": "Q2 – Dutugemunu",
        "context": (
            "දුටුගැමුණු රජු ක්‍රිස්තු පූර්ව 161 සිට 137 දක්වා රාජ්‍යය පාලනය කළේය. "
            "ඔහු එළාර රජු පරාජය කොට ශ්‍රී ලංකාව එකමුතු කළේය. "
            "රුවන්වැලිසාය ස්තූපය ඉදිකිරීම ආරම්භ කළේ දුටුගැමුණු රජු විසිනි. "
            "ඔහු අනුරාධපුර රාජධානියේ වීරත්වය ගත් රජකු ලෙස ඉතිහාසයේ සටහන් වෙයි."
        ),
        "question": "රුවන්වැලිසාය ස්තූපය ඉදිකිරීම ආරම්භ කළේ කවුද?",
        "expected_answer": "දුටුගැමුණු රජු",
    },
    {
        "name": "Q3 – Buddhism arrival",
        "context": (
            "බුදු දහම ශ්‍රී ලංකාවට ගෙන ආවේ මහින්ද තෙරුන් විසිනි. "
            "එය සිදු වූයේ ක්‍රිස්තු පූර්ව 247 දී දේවානම්පියතිස්ස රජුගේ පාලන සමයේදීය. "
            "මහින්ද තෙරුන් අශෝක අධිරාජයාගේ පුත්‍රයා බව සැලකේ. "
            "බුදු දහම ශ්‍රී ලංකාවේ රාජකීය ආගම බවට පත් වූයේ එතැන් සිටය."
        ),
        "question": "බුදු දහම ශ්‍රී ලංකාවට ගෙන ආවේ කවුද?",
        "expected_answer": "මහින්ද තෙරුන්",
    },
    {
        "name": "Q4 – Vijaya",
        "context": (
            "විජය කුමාරයා ශ්‍රී ලංකාවේ ඉතිහාසයේ සඳහන් පළමු රජු ලෙස සැලකේ. "
            "ඔහු ඉන්දියාවේ සිංහ රාජකුලයෙන් පැවත ආ බව මහාවංශයේ සඳහන් වේ. "
            "විජය ශ්‍රී ලංකාවට පැමිණියේ ක්‍රිස්තු පූර්ව 543 දී බව විශ්වාස කෙරේ. "
            "ඔහු සමඟ 700 දෙනෙකු ශ්‍රී ලංකාවට පැමිණි බව සඳහන් වේ."
        ),
        "question": "විජය ශ්‍රී ලංකාවට පැමිණියේ කවදාද?",
        "expected_answer": "ක්‍රිස්තු පූර්ව 543 දී",
    },
    {
        "name": "Q5 – British takeover",
        "context": (
            "බ්‍රිතාන්‍යයන් 1796 දී ලන්දේසීන්ගෙන් ශ්‍රී ලංකාවේ බලය අත්පත් කරගත්හ. "
            "1815 දී මහනුවර සම්මුතිය අත්සන් කිරීමෙන් මහනුවර රාජධානිය ද ඔවුන් යටතට පත් විය. "
            "බ්‍රිතාන්‍ය පාලනය යටතේ කෝපි, තේ සහ පොල් වගාව ප්‍රවර්ධනය විය. "
            "ශ්‍රී ලංකාව සිලෝන් නමින් බ්‍රිතාන්‍ය යටත් විජිතයක් ලෙස 1948 දක්වා පැවතිණි."
        ),
        "question": "මහනුවර සම්මුතිය අත්සන් කෙරුණේ කවදාද?",
        "expected_answer": "1815 දී",
    },
    {
        "name": "Q6 – Mahaweli irrigation",
        "context": (
            "මහවැලි සංවර්ධන ව්‍යාපෘතිය ශ්‍රී ලංකාවේ විශාලතම සංවර්ධන ව්‍යාපෘතියයි. "
            "එය 1970 දශකයේදී ජේ.ආර්. ජයවර්ධන ජනාධිපතිවරයාගේ නායකත්වය යටතේ ආරම්භ කෙරිණි. "
            "මෙම ව්‍යාපෘතිය හරහා විශාල ජල විදුලි බලශක්තියක් ජනනය වේ. "
            "වැව් 30කට අධික ප්‍රමාණයක් සහ ඇල මාර්ග ජාලයක් ගොඩනගන ලදී."
        ),
        "question": "මහවැලි සංවර්ධන ව්‍යාපෘතිය ආරම්භ කෙරුණේ කාගේ නායකත්වය යටතේද?",
        "expected_answer": "ජේ.ආර්. ජයවර්ධන ජනාධිපතිවරයාගේ නායකත්වය යටතේ",
    },
    {
        "name": "Q7 – Tooth Relic",
        "context": (
            "දළදා වහන්සේ ශ්‍රී ලංකාවට ගෙන ආවේ හේමමාලා කුමරිය සහ දන්ත කුමාරයා විසිනි. "
            "ඔවුන් එය ඔවුන්ගේ කෙස් රැළිය තුළ සඟවාගෙන ලංකාවට ගෙන ආහ. "
            "එය සිදු වූයේ ක්‍රිස්තු වර්ෂ 4 වැනි සියවසේදීය. "
            "දළදා මාලිගාව දැනට පිහිටා ඇත්තේ මහනුවර නගරයේය."
        ),
        "question": "දළදා වහන්සේ ශ්‍රී ලංකාවට ගෙන ආවේ කවුද?",
        "expected_answer": "හේමමාලා කුමරිය සහ දන්ත කුමාරයා",
    },
    {
        "name": "Q8 – First female PM",
        "context": (
            "සිරිමාවෝ බණ්ඩාරනායක මහත්මිය ලොව පළමු කාන්තා අගමැතිවරිය වූවාය. "
            "ඇය 1960 දී ශ්‍රී ලංකාවේ අගමැතිනිය ලෙස පත් වූවාය. "
            "ඇය ශ්‍රී ලංකා නිදහස් පක්ෂයේ නායිකාවක් ලෙස කටයුතු කළාය. "
            "ඇගේ ස්වාමිපුරුෂයා වූ එස්.ඩබ්.ආර්.ඩී. බණ්ඩාරනායක 1959 දී ඝාතනය කරන ලදී."
        ),
        "question": "සිරිමාවෝ බණ්ඩාරනායක මහත්මිය අගමැතිනිය ලෙස පත් වූයේ කවදාද?",
        "expected_answer": "1960 දී",
    },
    {
        "name": "Q9 – Unanswerable",
        "context": (
            "ගාල්ල කොටුව 1588 දී පෘතුගීසීන් විසින් ඉදිකරන ලදී. "
            "පසුව 1663 දී ලන්දේසීන් එය නවීකරණය කළහ. "
            "ගාල්ල කොටුව යූනෙස්කෝ ලෝක උරුම අඩවියක් ලෙස 1988 දී නම් කෙරිණි."
        ),
        "question": "ගාල්ල කොටුවේ ප්‍රධාන දොරටුවේ පළල කීයද?",
        "expected_answer": "නොදනී",
    },
    {
        "name": "Q10 – Mahinda Rajapaksa",
        "context": (
            "මහින්ද රාජපක්ෂ මහතා 2005 සිට 2015 දක්වා ශ්‍රී ලංකාවේ ජනාධිපතිවරයා ලෙස සේවය කළේය. "
            "ඔහුගේ පාලන සමයේදී 2009 මැයි මාසයේදී දශක තුනකට අධික ගැටුම අවසන් විය. "
            "ඔහු හම්බන්තොට දිස්ත්‍රික්කයෙන් පාර්ලිමේන්තු මන්ත්‍රීවරයෙකු ලෙස ද කටයුතු කළේය."
        ),
        "question": "ශ්‍රී ලංකාවේ සන්නද්ධ ගැටුම අවසන් වූයේ කවදාද?",
        "expected_answer": "2009 මැයි මාසයේදී",
    },
    {
        "name": "Q11 – Elara",
        "context": (
            "එළාර රජු දකුණු ඉන්දියාවේ චෝළ වංශයට අයත් රජකු ලෙස සැලකේ. "
            "ඔහු ක්‍රිස්තු පූර්ව 205 සිට 161 දක්වා අනුරාධපුරය පාලනය කළේය. "
            "පසුව දුටුගැමුණු රජු ඔහුව පරාජය කළේය. "
            "එළාර රජුගේ පාලනය සාධාරණ පාලනයක් ලෙස ද ජනප්‍රවාදයේ සඳහන් වේ."
        ),
        "question": "එළාර රජු පරාජය කළේ කවුද?",
        "expected_answer": "දුටුගැමුණු රජු",
    },
    {
        "name": "Q12 – First king of Sri Lanka",
        "context": (
            "විජය කුමාරයා ශ්‍රී ලංකාවේ ඉතිහාසයේ සඳහන් පළමු රජු ලෙස සැලකේ. "
            "ඔහුගේ පැමිණීමත් සමඟ නව රාජකීය වංශයක ආරම්භය සිදු වූ බව මහාවංශයේ සඳහන් වේ. "
            "විජය සමග පිරිසක් ද මෙරටට පැමිණියහ."
        ),
        "question": "ශ්‍රී ලංකාවේ පළමු රජු ලෙස සැලකෙන්නේ කවුද?",
        "expected_answer": "විජය කුමාරයා",
    },
    {
        "name": "Q13 – Galle Fort UNESCO",
        "context": (
            "ගාල්ල කොටුව 1588 දී පෘතුගීසීන් විසින් ඉදිකරන ලදී. "
            "පසුව 1663 දී ලන්දේසීන් එය නවීකරණය කළහ. "
            "ගාල්ල කොටුව යූනෙස්කෝ ලෝක උරුම අඩවියක් ලෙස 1988 දී නම් කෙරිණි."
        ),
        "question": "ගාල්ල කොටුව යූනෙස්කෝ ලෝක උරුම අඩවියක් ලෙස නම් කෙරුණේ කවදාද?",
        "expected_answer": "1988 දී",
    },
    {
        "name": "Q14 – British crops",
        "context": (
            "බ්‍රිතාන්‍ය පාලනය යටතේ කෝපි, තේ සහ පොල් වගාව ප්‍රවර්ධනය විය. "
            "මෙම වගාවන් තුළින් විශාල ආර්ථික වෙනස්කම් සිදුවිය. "
            "ශ්‍රී ලංකාවේ වතු ආර්ථිකය එහි ප්‍රතිඵලයක් ලෙස වර්ධනය විය."
        ),
        "question": "බ්‍රිතාන්‍ය පාලනය යටතේ ප්‍රවර්ධනය වූ වගාවන් මොනවාද?",
        "expected_answer": "කෝපි, තේ සහ පොල් වගාව",
    },
    {
        "name": "Q15 – Ashoka's son",
        "context": (
            "මහින්ද තෙරුන් බුදු දහම ශ්‍රී ලංකාවට ගෙන ආ පුද්ගලයා ලෙස සැලකේ. "
            "ඔහු අශෝක අධිරාජයාගේ පුත්‍රයා බව ඉතිහාසයේ සඳහන් වේ. "
            "ඔහු දේවානම්පියතිස්ස රජුගේ පාලන සමයේදී ශ්‍රී ලංකාවට පැමිණියේය."
        ),
        "question": "මහින්ද තෙරුන් කාගේ පුත්‍රයා ලෙස සැලකේද?",
        "expected_answer": "අශෝක අධිරාජයාගේ",
    },
    {
        "name": "Q16 – Kandyan Convention",
        "context": (
            "1815 දී මහනුවර සම්මුතිය අත්සන් කිරීමෙන් මහනුවර රාජධානිය බ්‍රිතාන්‍යයන් යටතට පත් විය. "
            "මෙම සම්මුතියෙන් මෙරට අභ්‍යන්තර භූමිය සම්පූර්ණයෙන්ම බ්‍රිතාන්‍ය පාලනයට පැමිණියේය."
        ),
        "question": "මහනුවර සම්මුතිය අත්සන් කෙරුණේ කවදාද?",
        "expected_answer": "1815 දී",
    },
    {
        "name": "Q17 – Sri Dalada Maligawa",
        "context": (
            "දළදා මාලිගාව දැනට පිහිටා ඇත්තේ මහනුවර නගරයේය. "
            "එය ශ්‍රී ලංකාවේ වැදගත්ම බෞද්ධ සිද්ධස්ථානයක් ලෙස සැලකේ."
        ),
        "question": "දළදා මාලිගාව පිහිටා ඇත්තේ කුමන නගරයේද?",
        "expected_answer": "මහනුවර නගරයේ",
    },
    {
        "name": "Q18 – Mahaweli infrastructure",
        "context": (
            "මහවැලි සංවර්ධන ව්‍යාපෘතිය යටතේ වැව් 30කට අධික ප්‍රමාණයක් සහ ඇල මාර්ග ජාලයක් ගොඩනගන ලදී. "
            "මෙය ජල සම්පාදනය හා වාරිමාර්ග ක්ෂේත්‍රයට විශාල බලපෑමක් කළේය."
        ),
        "question": "මහවැලි ව්‍යාපෘතියෙන් ගොඩනගන ලද යටිතල පහසුකම් දෙකක් සඳහන් කරන්න.",
        "expected_answer": "වැව් 30කට අධික ප්‍රමාණයක් සහ ඇල මාර්ග ජාලයක්",
    },
    {
        "name": "Q19 – Vijaya's companions",
        "context": (
            "විජය ශ්‍රී ලංකාවට පැමිණියේ ක්‍රිස්තු පූර්ව 543 දී බව විශ්වාස කෙරේ. "
            "ඔහු සමඟ 700 දෙනෙකු ශ්‍රී ලංකාවට පැමිණි බව සඳහන් වේ. "
            "මෙම සිදුවීම ශ්‍රී ලංකා ඉතිහාසයේ වැදගත් හැරවුම් ලක්ෂ්‍යයක් ලෙස සැලකේ."
        ),
        "question": "විජය සමඟ ශ්‍රී ලංකාවට පැමිණි පිරිස කොපමණද?",
        "expected_answer": "700 දෙනෙකු",
    },
    {
        "name": "Q20 – Galle Fort gate",
        "context": (
            "ගාල්ල කොටුව 1588 දී පෘතුගීසීන් විසින් ඉදිකරන ලදී. "
            "පසුව 1663 දී ලන්දේසීන් එය නවීකරණය කළහ. "
            "ගාල්ල කොටුව යූනෙස්කෝ ලෝක උරුම අඩවියක් ලෙස 1988 දී නම් කෙරිණි."
        ),
        "question": "ගාල්ල කොටුවේ ප්‍රධාන දොරටුවේ පළල කීයද?",
        "expected_answer": "නොදනී",
    },
]

print("\n" + "=" * 60)
print("20 HISTORY QA TESTS")
print("=" * 60)

correct = 0

for test in quick_tests:
    print(f"\n📌 [{test['name']}]")
    print(f"   Q: {test['question']}")

    pred = run_qa(test["context"], test["question"])
    gold = test["expected_answer"]

    ok = validate_answer(pred, gold)
    correct += int(ok)

    status = "✅ PASS" if ok else "❌ FAIL"
    print(f"   A: {pred}")
    print(f"   G: {gold}")
    print(f"   {status}")
    print("-" * 60)

print(f"\nFinal Score: {correct}/{len(quick_tests)}")
print(f"Accuracy: {(correct / len(quick_tests)) * 100:.1f}%")


20 HISTORY QA TESTS

📌 [Q1 – Parakramabahu]
   Q: පරාක්‍රම සමුද්‍රය ඉදිකළේ කවුද?
   A: ඔහු ය.]
   G: පරාක්‍රමබාහු පළමුවැනි රජු
   ❌ FAIL
------------------------------------------------------------

📌 [Q2 – Dutugemunu]
   Q: රුවන්වැලිසාය ස්තූපය ඉදිකිරීම ආරම්භ කළේ කවුද?
   A: දුටුගැමුණු රජු විසිනි.]
   G: දුටුගැමුණු රජු
   ✅ PASS
------------------------------------------------------------

📌 [Q3 – Buddhism arrival]
   Q: බුදු දහම ශ්‍රී ලංකාවට ගෙන ආවේ කවුද?
   A: මහින්ද හිමියන් ය.]
   G: මහින්ද තෙරුන්
   ✅ PASS
------------------------------------------------------------

📌 [Q4 – Vijaya]
   Q: විජය ශ්‍රී ලංකාවට පැමිණියේ කවදාද?
   A: ක්‍රිස්තු පූර්ව 543 දී ය.]
   G: ක්‍රිස්තු පූර්ව 543 දී
   ✅ PASS
------------------------------------------------------------

📌 [Q5 – British takeover]
   Q: මහනුවර සම්මුතිය අත්සන් කෙරුණේ කවදාද?
   A: 1796 වර්ෂයේදී ය.]
   G: 1815 දී
   ❌ FAIL
------------------------------------------------------------

📌 [Q6 – Mahaweli irrigation]
   Q: මහවැලි සංවර්ධන ව්

In [23]:
repo_id = "isji/sinllama-1b-qa"

trainer.model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print(f"Adapter pushed to https://huggingface.co/{repo_id}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Adapter pushed to https://huggingface.co/isji/sinllama-1b-qa
